# Yield Curve Bootstrapping

Module: Term Structure and Interest Rate Models

## Lesson summary

A yield curve describes how interest rates vary across maturities. Fixed-income valuation becomes more realistic when each cash flow is discounted with a maturity-specific discount factor instead of a single flat yield.

This lesson introduces the relationship between discount factors, spot rates, forward rates, and par rates, then builds a simple bootstrapping workflow.

## Learning objectives

By the end of this lesson, students should be able to:

- convert between discount factors and spot rates;
- bootstrap discount factors from simple market instruments;
- derive implied forward rates;
- plot and interpret the shape of a yield curve;
- explain the limitations of curve construction with sparse inputs.

## Core relationships

The discount factor for maturity $T$ under annual compounding is:

$$
D(0,T) = \frac{1}{(1+s_T)^T},
$$

where $s_T$ is the spot rate for maturity $T$.

The annual spot rate implied by a discount factor is:

$$
s_T = D(0,T)^{-1/T} - 1.
$$

The one-period forward rate between $T_1$ and $T_2$ is:

$$
f(T_1,T_2) =
\left(\frac{D(0,T_1)}{D(0,T_2)}\right)^{1/(T_2-T_1)} - 1.
$$

## Python setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Synthetic market instruments

In [ ]:
market = pd.DataFrame(
    {
        "maturity": [0.5, 1.0, 1.5, 2.0, 3.0],
        "coupon_rate": [0.00, 0.00, 0.045, 0.050, 0.055],
        "price": [98.20, 95.90, 100.10, 100.25, 100.40],
        "frequency": [2, 2, 2, 2, 2],
    }
)
market

## Bootstrap discount factors

In [ ]:
def bootstrap_discount_factors(instruments, face_value=100):
    discount_factors = {}

    for _, row in instruments.sort_values("maturity").iterrows():
        maturity = row["maturity"]
        frequency = int(row["frequency"])
        periods = int(round(maturity * frequency))
        coupon = face_value * row["coupon_rate"] / frequency

        known_pv = 0.0
        for period in range(1, periods):
            payment_time = period / frequency
            if payment_time in discount_factors:
                known_pv += coupon * discount_factors[payment_time]

        final_cash_flow = face_value + coupon
        discount_factors[maturity] = (row["price"] - known_pv) / final_cash_flow

    curve = pd.DataFrame(
        {
            "maturity": list(discount_factors.keys()),
            "discount_factor": list(discount_factors.values()),
        }
    ).sort_values("maturity")
    curve["spot_rate"] = curve["discount_factor"] ** (-1 / curve["maturity"]) - 1
    return curve


curve = bootstrap_discount_factors(market)
curve

## Forward rates

In [ ]:
curve["forward_rate"] = np.nan
for i in range(1, len(curve)):
    d1 = curve.loc[i - 1, "discount_factor"]
    d2 = curve.loc[i, "discount_factor"]
    t1 = curve.loc[i - 1, "maturity"]
    t2 = curve.loc[i, "maturity"]
    curve.loc[i, "forward_rate"] = (d1 / d2) ** (1 / (t2 - t1)) - 1

curve

## Visualizing the curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curve["maturity"], curve["spot_rate"], marker="o", label="Spot rate")
ax.plot(curve["maturity"], curve["forward_rate"], marker="s", label="Forward rate")
ax.set_xlabel("Maturity")
ax.set_ylabel("Rate")
ax.set_title("Bootstrapped Spot and Forward Rates")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Model limitations

- Bootstrapped curves inherit quote noise, missing maturities, and instrument convention errors.
- Interpolation choices can affect forward rates even when spot rates look smooth.
- A curve built from simplified instruments should not be used as a production discounting curve.